In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import math
import tifffile
import seaborn as sns
import numpy as np
import re

In [ ]:
# -----------------------------
# Paths
# -----------------------------

BASE_DIR = Path.cwd().parents[1]

METADATA_FILE = (
    BASE_DIR
    / "FIB_SEM_optimization"
    / "FIB_SEM_Opt_datasets.csv"
)

COMPARISON_DIR = (
    BASE_DIR
    / "FIB_SEM_optimization"
    / "comparison_figures"
    / "segmentation_quality"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------
# Load metadata once
# -----------------------------

metadata = pd.read_csv(
    METADATA_FILE
)

quality_results = pd.read_csv(
    BASE_DIR / "FIB_SEM_optimization" / "results" / "image_quality_results.csv"
)
quality_results_normalised = pd.read_csv(
    BASE_DIR / "FIB_SEM_optimization" / "results" / "image_quality_results_normalised.csv"
)


# -----------------------------
# Extract image ID
# -----------------------------

def get_image_id(path):

    match = re.search(
        r"_LFP_(\d+)(?:_mask)?$",
        path.stem
    )

    if match:
        return int(match.group(1))

    return None

In [ ]:
def plot_segmented_images(
    param,
    metadata_filter,
    title="Segmented FIB-SEM Images",
    filename="segmented_comparison",
    cols=4,
    suction_voltage=None,
    signal_out=None,
    Mode=None,
):

    """
    Plot and save segmented FIB-SEM images based on metadata filters.

    Example:

    plot_segmented_images(
        param="Param_2",
        metadata_filter=(
            (metadata["parameter_set"] == 2) &
            (metadata["current_nA"] == 0.1)
        ),
        title="Parameter Set 2 | Current 0.1 nA",
        filename="ParameterSet_2_Current_0.1nA_Segmented"
    )
    """

    # --------------------------------
    # Segmented image directory
    # --------------------------------

    BASE_DIR = Path.cwd().parents[1]

    MASK_DIR = (
        BASE_DIR
        / "FIB_SEM_optimization"
        / "images"
        / "segmented_masks"
        / param
    )


    # --------------------------------
    # Select metadata
    # --------------------------------

    selected = metadata[
        metadata_filter
    ].sort_values(
        "image_id"
    )


    if len(selected) == 0:
        print("No images found")
        return


    wanted_ids = set(
        selected["image_id"]
    )


    # --------------------------------
    # Find matching segmented images
    # --------------------------------

    image_paths = []

    for image_file in MASK_DIR.glob("*_mask.tif"):

        image_id = get_image_id(
            image_file
        )

        if image_id in wanted_ids:
            image_paths.append(
                image_file
            )


    image_paths = sorted(
        image_paths,
        key=get_image_id
    )


    n_images = len(image_paths)


    if n_images == 0:
        print("No segmented TIFF files found")
        return


    # --------------------------------
    # Figure layout
    # --------------------------------

    rows = math.ceil(
        n_images / cols
    )

    plt.close("all")

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(30, 8 * rows),
        dpi=200
    )


    if n_images == 1:
        axes = [axes]

    else:
        axes = axes.flatten()


    # --------------------------------
    # Plot segmented images
    # --------------------------------

    for ax, image_path in zip(
        axes,
        image_paths
    ):

        image = tifffile.imread(
            image_path
        )


        ax.imshow(
            image,
            cmap="gray",
            interpolation="nearest",
            aspect="equal"
        )


        image_id = get_image_id(
            image_path
        )


        meta_row = selected[
            selected["image_id"] == image_id
        ].iloc[0]


        ax.set_title(
            image_path.stem,
            fontsize=16
        )


        info = (
            f"ID: {image_id}\n"
            + (
                f"Suction Tube Voltage: {meta_row.get('suction_V', 'N/A')}\n"
                if suction_voltage is not None
                else ""
            )
            + (
                f"Mode: {meta_row.get('mode', 'N/A')}\n"
                if Mode is not None
                else ""
            )
            + f"Detector: {meta_row.get('detector', 'N/A')}\n"
            + (
                f"Signal Out: {meta_row.get('signal_out', 'N/A')}\n"
                if signal_out is not None
                else ""
            )
            + f"Current: {meta_row.get('current_nA', 'N/A')} nA\n"
            + f"Voltage: {meta_row.get('voltage_kV', 'N/A')} kV\n"
            + f"Magnification: {meta_row.get('magnification', 'N/A')}\n"
            + (
                f"Notes: {meta_row['notes']}"
                if pd.notna(meta_row["notes"])
                and str(meta_row["notes"]).strip().upper() != "NA"
                else ""
            )
        )


        ax.text(
            0.5,
            -0.15,
            info,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=12
        )


        ax.axis("off")


    # --------------------------------
    # Remove empty axes
    # --------------------------------

    for ax in axes[n_images:]:
        ax.axis("off")


    # --------------------------------
    # Figure title
    # --------------------------------

    plt.suptitle(
        title,
        fontsize=16
    )


    plt.tight_layout()


    # --------------------------------
    # Save figure
    # --------------------------------

    save_path = (
        COMPARISON_DIR
        / f"{filename}.png"
    )


    try:

        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

    finally:

        plt.close(fig)


    print(
        f"Displayed {n_images} segmented images"
    )

    print(
        f"Saved: {save_path}"
    )

In [ ]:
def plot_segmented_masks_comparison(
    param,
    metadata_filter,
    row_variable,
    column_variable,
    row_order=None,
    column_order=None,
    column_map=None,
    title="Segmented Mask Comparison",
    filename="segmented_mask_comparison"
):

    # ------------------------------------------------------
    # Mask directory
    # ------------------------------------------------------

    MASK_DIR = (
        BASE_DIR
        / "FIB_SEM_optimization"
        / "images"
        / "segmented_masks"
        / param
    )


    # ------------------------------------------------------
    # Select metadata
    # ------------------------------------------------------

    selected = metadata[
        metadata_filter
    ].copy()


    if selected.empty:
        print("No matching images found.")
        return


    # ------------------------------------------------------
    # Ordering rows
    # ------------------------------------------------------

    if row_order is None:

        row_values = sorted(
            selected[row_variable].unique()
        )

    else:

        row_values = row_order


    # ------------------------------------------------------
    # Ordering columns
    # ------------------------------------------------------

    if column_order is None:

        column_values = sorted(
            selected[column_variable].unique()
        )

    else:

        column_values = column_order


    # ------------------------------------------------------
    # Determine number of columns
    # ------------------------------------------------------

    if column_map is not None:

        max_columns = max(
            len(values)
            for values in column_map.values()
        )

    else:

        max_columns = len(column_values)


    # ------------------------------------------------------
    # Create figure
    # ------------------------------------------------------

    plt.close("all")

    fig, axes = plt.subplots(

        len(row_values),

        max_columns,

        figsize=(
            5 * max_columns,
            6 * len(row_values)
        ),

        dpi=400

    )


    # ------------------------------------------------------
    # Handle axes shape
    # ------------------------------------------------------

    if len(row_values) == 1 and max_columns == 1:

        axes = [[axes]]

    elif len(row_values) == 1:

        axes = [axes]

    elif max_columns == 1:

        axes = [[ax] for ax in axes]


    # ------------------------------------------------------
    # Fill figure
    # ------------------------------------------------------

    for row_index, row_value in enumerate(row_values):


        # Select columns for this row
        if column_map is not None:

            current_columns = column_map[row_value]

        else:

            current_columns = column_values


        for col_index, col_value in enumerate(current_columns):


            ax = axes[row_index][col_index]


            subset = selected[

                (selected[row_variable] == row_value)

                &

                (selected[column_variable] == col_value)

            ]


            if subset.empty:

                ax.axis("off")
                continue


            meta = subset.iloc[0]


            image_id = meta["image_id"]


            # --------------------------------------------------
            # Find segmented mask
            # --------------------------------------------------

            image_path = None


            for p in MASK_DIR.glob("*_mask.tif"):

                if get_image_id(p) == image_id:

                    image_path = p
                    break


            if image_path is None:

                ax.axis("off")
                continue


            # --------------------------------------------------
            # Load mask
            # --------------------------------------------------

            mask = tifffile.imread(
                image_path
            )


            ax.imshow(

                mask,

                cmap="gray",

                interpolation="nearest",

                aspect="equal"

            )


            # --------------------------------------------------
            # Column heading
            # --------------------------------------------------

            if row_index == 0 or column_map is not None:

                ax.set_title(

                    f"{column_variable}\n{col_value}",

                    fontsize=14,

                    fontweight="bold"

                )


            # --------------------------------------------------
            # Row heading
            # --------------------------------------------------

            if col_index == 0:

                ax.set_ylabel(

                    f"{row_variable}\n{row_value}",

                    fontsize=14,

                    fontweight="bold",

                    rotation=0,

                    labelpad=55,

                    va="center"

                )


            # --------------------------------------------------
            # Metadata
            # --------------------------------------------------

            info = (

                f"ID: {image_id}\n"

                f"Detector: {meta.get('detector','N/A')}\n"

                + (

                    f"Mode: {meta.get('mode','N/A')}\n"

                    if pd.notna(meta.get('mode'))

                    and str(meta.get('mode')).strip().upper() != "NA"

                    else ""
                )

                + f"Current: {meta.get('current_nA','N/A')} nA\n"

                + f"Voltage: {meta.get('voltage_kV','N/A')} kV\n"

                + f"Magnification: {meta.get('magnification','N/A')}"
            )


            ax.text(

                0.5,

                -0.10,

                info,

                transform=ax.transAxes,

                ha="center",

                va="top",

                fontsize=11

            )


            ax.set_xticks([])
            ax.set_yticks([])


    # ------------------------------------------------------
    # Hide unused axes
    # ------------------------------------------------------

    for row in axes:

        for ax in row:

            if not ax.has_data():

                ax.axis("off")


    # ------------------------------------------------------
    # Figure title
    # ------------------------------------------------------

    plt.suptitle(

        title,

        fontsize=18,

        fontweight="bold"

    )


    plt.tight_layout()


    # ------------------------------------------------------
    # Save
    # ------------------------------------------------------

    save_path = (

        COMPARISON_DIR

        / f"{filename}.png"

    )


    try:

        plt.savefig(

            save_path,

            dpi=400,

            bbox_inches="tight"

        )

    finally:

        plt.close(fig)


    print(
        f"Saved to:\n{save_path}"
    )

In [ ]:
plot_segmented_images(
    param="Param_1",
    metadata_filter=(
        (metadata["parameter_set"] == 1)
    ),
    title="Parameter Set 1 | Current comparison | Segmented",
    filename="Parameter_Set_1_Current_Comparison_Segmented"
)

In [ ]:
plot_segmented_masks_comparison(
    param="Param_2",

    metadata_filter=
        metadata["parameter_set"] == 2,

    row_variable="current_nA",

    column_variable="voltage_kV",

    row_order=[0.1, 0.0031],

    column_order=[5, 4, 3, 2, 1],

    title="Parameter Set 2\nSegmented Masks",

    filename="Parameter_Set_2_Segmented_Masks"

)

In [ ]:
plot_segmented_images(
    param="Param_3",
    metadata_filter=(metadata["parameter_set"] == 3),
    title="Parameter Set 3 | Tube Voltage Comparison",
    filename="Parameter_Set_3_Tube_Voltage_Comparison",
    suction_voltage=True
)

In [ ]:
plot_segmented_masks_comparison(
    param="Param_4",

    metadata_filter=
        metadata["parameter_set"] == 4,

    row_variable="magnification",

    column_variable="voltage_kV",

    row_order=["50000x", "100000x"],

    column_order=[5, 10, 15, 20],

    title="Parameter Set 4\nVoltage & Magnification Comparison",

    filename="Parameter_Set_4_Voltage_Magnification_Comparison"

)

In [ ]:
plot_segmented_images(
    param= "Param_5",
    metadata_filter= (metadata["parameter_set"] == 5),
    title="Parameter Set 5\nCurrent Comparison",
    filename="Parameter_Set_5_Current_Comparison",
    signal_out=True,
)

In [ ]:
voltage_current_map = {

    30: [0.04, 0.0011, 0.0077, 0.024],

    15: [0.022, 0.00096, 0.0038, 0.011]

}


plot_segmented_masks_comparison(
    param="Param_6",

    metadata_filter=(
        metadata["parameter_set"] == 6
    ),

    row_variable="voltage_kV",

    column_variable="current_nA",

    row_order=[
        30,
        15
    ],

    column_map=voltage_current_map,

    title="Parameter Set 6\nVoltage & Current Comparison",

    filename="Parameter_Set_6_Voltage_Current_Comparison"

)

In [ ]:
plot_segmented_images(
    param= "Param_7",
    metadata_filter= (metadata["parameter_set"] == 7),
    title="Parameter Set 7\nSame settings but different areas",
    filename="Parameter_Set_7_area_Comparison",
    signal_out=True,
)

In [ ]:
plot_segmented_masks_comparison(
    param="Param_8",

    metadata_filter=
    (metadata["parameter_set"] == 8) & (metadata["voltage_kV"] == 5) & (metadata["image_id"].between(66, 69)),
    row_variable="voltage_kV",

    column_variable="current_nA",

    row_order=[5],

    column_order=[0.2, 0.4, 0.8, 3.2],

    title="Parameter Set 8\n ETD Current comparison",

    filename="Parameter_Set_8_ETD_Current_Comparison"

)

In [ ]:
plot_segmented_masks_comparison(
    param="Param_8",

    metadata_filter=
    (metadata["parameter_set"] == 8),
    row_variable="current_nA",

    column_variable="voltage_kV",

    row_order=[0.8],

    column_order=[5, 10, 15, 20, 30],

    title="Parameter Set 8\n ETD Voltage comparison at 0.8 nA",

    filename="Parameter_Set_8_ETD_Voltage_Comparison_0.8nA"

)

In [ ]:
plot_segmented_masks_comparison(
    param="Param_8",

    metadata_filter=
    (metadata["parameter_set"] == 8),
    row_variable="current_nA",

    column_variable="voltage_kV",

    row_order=[1.6],

    column_order=[5, 10, 15, 20, 30],

    title="Parameter Set 8\n ETD Voltage comparison at 1.6 nA",

    filename="Parameter_Set_8_ETD_Voltage_Comparison_1.6nA"

)

In [ ]:
plot_segmented_masks_comparison(
    param="Param_8",

    metadata_filter=
    (metadata["parameter_set"] == 8),
    row_variable="current_nA",

    column_variable="voltage_kV",

    row_order=[0.4],

    column_order=[5, 10, 15, 20, 30],

    title="Parameter Set 8\n ETD Voltage comparison at 0.4 nA",

    filename="Parameter_Set_8_ETD_Voltage_Comparison_0.4nA"

)

In [ ]:
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt

def plot_seg_mask_overlay(param):
    """
    Plot original images, segmentation masks, and overlays for a given parameter set.

    Example:
        plot_seg_mask_overlay("Param_1")
    """
    BASE_DIR = Path.cwd().parents[1]

    MASK_DIR = (
        BASE_DIR
        / "FIB_SEM_optimization"
        / "images"
        / "segmented_masks"
        / param
    )

    IMAGE_DIR = (
        BASE_DIR
        / "FIB_SEM_optimization"
        / "images"
        / "cropped"
        / param
    )


    mask_files = sorted(
        MASK_DIR.glob("*_mask.tif")
    )


    n_images = len(mask_files)


    # Three columns:
    # Original | Mask | Overlay
    cols = 3
    rows = n_images


    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(15, 4 * rows)
    )


    # Handle single image case
    if n_images == 1:
        axes = axes.reshape(1, -1)


    for i, mask_path in enumerate(mask_files):

        # Load mask
        mask = tifffile.imread(
            mask_path
        )


        # Find original image
        image_path = (
            IMAGE_DIR
            / mask_path.name.replace(
                "_mask",
                ""
            )
        )


        image = tifffile.imread(
            image_path
        )


        # -----------------------
        # Original image
        # -----------------------
        axes[i, 0].imshow(
            image,
            cmap="gray"
        )

        axes[i, 0].set_title(
            f"{mask_path.stem.replace('_mask','')} \nOriginal",
            fontsize=9
        )

        axes[i, 0].axis("off")


        # -----------------------
        # Segmentation mask
        # -----------------------
        axes[i, 1].imshow(
            mask,
            cmap="gray"
        )

        axes[i, 1].set_title(
            "Segmentation Mask",
            fontsize=9
        )

        axes[i, 1].axis("off")


        # -----------------------
        # Overlay
        # -----------------------
        axes[i, 2].imshow(
            image,
            cmap="gray"
        )

        axes[i, 2].imshow(
            mask,
            cmap="jet",
            alpha=0.35
        )

        axes[i, 2].set_title(
            "Overlay",
            fontsize=9
        )

        axes[i, 2].axis("off")


    plt.tight_layout()
    plt.show()

In [ ]:
plot_seg_mask_overlay("Param_4")

In [ ]:
def process_quality_results_filtered(
    param,
    heatmap_group=None,
    filters=None,
    show_plots=True,
    title=None,
    filename=None,
):
    """
    Merge image-quality and segmentation results for a parameter set,
    save the merged CSV, optionally filter the results, and generate
    image-quality and segmentation-quality heatmaps.

    Parameters
    ----------
    param : str
        Parameter-set folder name.

        Example:
            "Param_1"
            "Param_2"
            "Param_5"

    heatmap_group : str, list of str, or None
        Column(s) used to group the heatmap.

        Examples:
            heatmap_group="current_nA"

            heatmap_group="voltage_kV"

            heatmap_group=["current_nA", "voltage_kV"]

        If None, no grouping is applied.

    filters : dict or None
        Filters used to select specific rows for the heatmaps
        and normalisation.

        Examples:

            filters={"voltage_kV": 5}

            filters={
                "voltage_kV": 5,
                "current_nA": 0.8
            }

            filters={
                "voltage_kV": [5, 10, 15]
            }

            filters={
                "voltage_kV": [5, 10],
                "current_nA": [0.1, 0.8]
            }

        If None, all merged results are used.

    show_plots : bool
        If True, display the heatmaps.
        If False, only save the heatmaps.

    title : str or None
        Optional custom base title for the heatmaps.

        If None, default titles are used:

            "Normalized Image Quality Metrics - Param_1"

            "Normalized Segmentation Quality Metrics - Param_1"

        If specified, "_Segmentation Quality" is automatically added
        to the segmentation heatmap title.

        Example:

            title="Parameter Set 2 - 5 kV"

        Produces:

            "Parameter Set 2 - 5 kV"

            "Parameter Set 2 - 5 kV - Segmentation Quality"

    filename : str or None
        Optional custom base filename for the heatmaps.

        If None, default filenames are used:

            image_quality_heatmap_param_1.png

            segmentation_quality_heatmap_param_1.png

        If specified:

            filename="param_2_5kv"

        Produces:

            param_2_5kv.png

            param_2_5kv_segmentation.png

    Returns
    -------
    quality_and_segmentation : pandas.DataFrame
        All merged image-quality and segmentation results.

    filtered_results : pandas.DataFrame
        Results remaining after applying filters.

    image_quality_norm : pandas.DataFrame
        Min-max normalised image-quality metrics based on
        the filtered results.

    segmentation_norm : pandas.DataFrame
        Min-max normalised segmentation metrics based on
        the filtered results.
    """

    # ========================================================
    # 1. File paths
    # ========================================================

    results_dir = (
        BASE_DIR
        / "FIB_SEM_optimization"
        / "results"
        / param
    )

    param_name = param.lower()


    # --------------------------------------------------------
    # Default filenames
    # --------------------------------------------------------

    if filename is None:

        image_quality_heatmap = (
            results_dir
            / f"image_quality_heatmap_{param_name}.png"
        )

        segmentation_quality_heatmap = (
            results_dir
            / f"segmentation_quality_heatmap_{param_name}.png"
        )

    else:

        filename = str(filename)

        # Remove .png if user supplied it
        if filename.lower().endswith(".png"):
            filename = filename[:-4]

        image_quality_heatmap = (
            results_dir
            / f"{filename}.png"
        )

        segmentation_quality_heatmap = (
            results_dir
            / f"{filename}_segmentation.png"
        )


    # --------------------------------------------------------
    # Merged CSV
    # --------------------------------------------------------

    seg_file = (
        results_dir
        / f"segmentation_quality_{param_name}.csv"
    )

    merged_file = (
        results_dir
        / f"merged_quality_results_{param_name}.csv"
    )


    # ========================================================
    # 2. Default titles
    # ========================================================

    if title is None:

        image_quality_title = (
            f"Normalized Image Quality Metrics - {param}"
        )

        segmentation_title = (
            f"Normalized Segmentation Quality Metrics - {param}"
        )

    else:

        image_quality_title = str(title)

        segmentation_title = (
            f"{title} - Segmentation Quality"
        )


    # ========================================================
    # 3. Metrics
    # ========================================================

    image_quality_metrics = [
        "Entropy",
        "Sharpness",
        "Gradient",
        "RMS_Contrast",
        "SNR"
    ]

    segmentation_metrics = [
        "threshold",
        "segmented_fraction",
        "mean_region_area",
        "median_region_area",
        "separability",
        "connected_components",
        "threshold_instability"
    ]

    # Threshold is kept in the merged data,
    # but excluded from the segmentation heatmap.

    segmentation_heatmap_metrics = [
        metric
        for metric in segmentation_metrics
        if metric != "threshold"
    ]


    # ========================================================
    # 4. Check segmentation file
    # ========================================================

    if not seg_file.exists():

        raise FileNotFoundError(
            f"Segmentation results file not found:\n{seg_file}"
        )


    # ========================================================
    # 5. Load segmentation results
    # ========================================================

    seg_results = pd.read_csv(seg_file)


    # ========================================================
    # 6. Check required image-quality columns
    # ========================================================

    required_quality_columns = [
        "detector",
        "sample_state",
        "material",
        "image_id"
    ] + image_quality_metrics

    missing_quality_columns = [
        column
        for column in required_quality_columns
        if column not in quality_results.columns
    ]

    if missing_quality_columns:

        raise ValueError(
            "Missing columns in quality_results:\n"
            f"{missing_quality_columns}"
        )


    # ========================================================
    # 7. Check required segmentation columns
    # ========================================================

    required_segmentation_columns = [
        "detector",
        "sample_state",
        "material",
        "image_id"
    ] + segmentation_metrics

    missing_segmentation_columns = [
        column
        for column in required_segmentation_columns
        if column not in seg_results.columns
    ]

    if missing_segmentation_columns:

        raise ValueError(
            "Missing columns in segmentation results:\n"
            f"{missing_segmentation_columns}"
        )


    # ========================================================
    # 8. Columns used to identify the same image
    # ========================================================

    merge_columns = [
        "detector",
        "sample_state",
        "material",
        "image_id"
    ]

    print("\n========================================")
    print(f"Processing {param}")
    print("========================================")

    print("\nMerge columns:")
    print(merge_columns)


    # ========================================================
    # 9. Clean merge columns
    # ========================================================

    for column in merge_columns:

        quality_results[column] = (
            quality_results[column]
            .astype(str)
            .str.strip()
        )

        seg_results[column] = (
            seg_results[column]
            .astype(str)
            .str.strip()
        )


    # --------------------------------------------------------
    # Keep image_id as float
    # --------------------------------------------------------

    quality_results["image_id"] = pd.to_numeric(
        quality_results["image_id"],
        errors="coerce"
    )

    seg_results["image_id"] = pd.to_numeric(
        seg_results["image_id"],
        errors="coerce"
    )


    # ========================================================
    # 10. Diagnostic information
    # ========================================================

    print("\nQuality results:")
    print(
        f"Rows: {len(quality_results)}"
    )

    print("\nSegmentation results:")
    print(
        f"Rows: {len(seg_results)}"
    )


    # ========================================================
    # 11. Merge image-quality and segmentation results
    # ========================================================

    seg_columns = (
        merge_columns
        + segmentation_metrics
    )

    quality_and_segmentation = quality_results.merge(
        seg_results[seg_columns],
        on=merge_columns,
        how="inner"
    )


    # ========================================================
    # 12. Check merge result
    # ========================================================

    if quality_and_segmentation.empty:

        print("\nWARNING: No images were matched.")

        print("\nUnique values in merge columns:")

        for column in merge_columns:

            quality_values = (
                quality_results[column]
                .drop_duplicates()
                .tolist()
            )

            segmentation_values = (
                seg_results[column]
                .drop_duplicates()
                .tolist()
            )

            common_values = (
                set(quality_values)
                & set(segmentation_values)
            )

            print(f"\n{column}")

            print(
                "Quality:",
                quality_values[:10]
            )

            print(
                "Segmentation:",
                segmentation_values[:10]
            )

            print(
                "Common:",
                list(common_values)[:10]
            )

        raise ValueError(
            f"No images could be merged for {param}.\n"
            "Check detector, sample_state, material and image_id."
        )


    # ========================================================
    # 13. Sort images by image ID
    # ========================================================

    quality_and_segmentation = (
        quality_and_segmentation
        .sort_values("image_id")
        .reset_index(drop=True)
    )


    # ========================================================
    # 14. Save ALL merged results
    # ========================================================

    quality_and_segmentation.to_csv(
        merged_file,
        index=False
    )

    print("\nMerged results saved to:")
    print(merged_file)

    print(
        "\nNumber of merged images:",
        len(quality_and_segmentation)
    )


    # ========================================================
    # 15. Apply filters
    # ========================================================

    filtered_results = (
        quality_and_segmentation.copy()
    )


    if filters is not None:

        print("\n========================================")
        print("Applying filters")
        print("========================================")

        for column, value in filters.items():

            # ------------------------------------------------
            # Check column exists
            # ------------------------------------------------

            if column not in filtered_results.columns:

                raise ValueError(
                    f"Filter column '{column}' not found."
                )


            # ------------------------------------------------
            # Multiple values
            # ------------------------------------------------

            if isinstance(
                value,
                (list, tuple, set)
            ):

                filtered_results = filtered_results[
                    filtered_results[column].isin(value)
                ]


            # ------------------------------------------------
            # Single value
            # ------------------------------------------------

            else:

                filtered_results = filtered_results[
                    filtered_results[column] == value
                ]


            print(
                f"{column} = {value}"
            )

            print(
                "Remaining rows:",
                len(filtered_results)
            )


    # ========================================================
    # 16. Check filtered results
    # ========================================================

    if filtered_results.empty:

        raise ValueError(
            "\nNo images remain after applying filters:\n"
            f"{filters}"
        )


    print(
        "\nNumber of images used for heatmaps:",
        len(filtered_results)
    )


    # ========================================================
    # 17. Normalisation function
    # ========================================================

    def min_max_normalize(series):

        min_value = series.min()
        max_value = series.max()

        if max_value == min_value:

            return pd.Series(
                1.0,
                index=series.index
            )

        return (
            series - min_value
        ) / (
            max_value - min_value
        )


    # ========================================================
    # 18. Set image ID as index
    # ========================================================

    results_indexed = (
        filtered_results
        .set_index("image_id")
    )


    # ========================================================
    # 19. Normalise image-quality metrics
    # ========================================================

    image_quality_norm = (
        results_indexed[
            image_quality_metrics
        ]
        .apply(min_max_normalize)
    )


    # ========================================================
    # 20. Normalise segmentation metrics
    # ========================================================

    segmentation_norm = (
        results_indexed[
            segmentation_metrics
        ]
        .apply(min_max_normalize)
    )


    # ========================================================
    # 21. Process heatmap grouping
    # ========================================================

    if heatmap_group is None:

        grouping_columns = []

    elif isinstance(heatmap_group, str):

        grouping_columns = [
            heatmap_group
        ]

    elif isinstance(heatmap_group, (list, tuple)):

        grouping_columns = list(
            heatmap_group
        )

    else:

        raise TypeError(
            "heatmap_group must be a string, "
            "list, tuple, or None."
        )


    # ========================================================
    # 22. Check grouping columns
    # ========================================================

    missing_group_columns = [
        column
        for column in grouping_columns
        if column not in filtered_results.columns
    ]

    if missing_group_columns:

        raise ValueError(
            "Heatmap grouping column(s) not found:\n"
            f"{missing_group_columns}"
        )


    # ========================================================
    # 23. Heatmap helper
    # ========================================================

    def save_heatmap(
        data,
        metrics,
        heatmap_title,
        heatmap_filename,
        grouping_columns=None
    ):

        plot_data = data.copy()


        # ----------------------------------------------------
        # Apply grouping
        # ----------------------------------------------------

        if grouping_columns:

            group_data = (
                filtered_results
                .set_index("image_id")[
                    grouping_columns
                ]
            )


            for column in grouping_columns:

                plot_data[
                    f"_group_{column}"
                ] = group_data[column]


            sort_columns = [
                f"_group_{column}"
                for column in grouping_columns
            ]


            plot_data = (
                plot_data
                .sort_values(sort_columns)
            )


        # ----------------------------------------------------
        # Create figure
        # ----------------------------------------------------

        figure_height = max(
            6,
            len(plot_data) * 0.35
        )


        plt.figure(
            figsize=(
                16,
                figure_height
            )
        )


        # ----------------------------------------------------
        # Draw heatmap
        # ----------------------------------------------------

        ax = sns.heatmap(
            plot_data[metrics],
            annot=True,
            fmt=".2f",
            cmap="viridis",
            vmin=0,
            vmax=1,
            linewidths=0.5,
            cbar_kws={
                "label": "Normalized Score"
            }
        )


        # ----------------------------------------------------
        # Add group separators
        # ----------------------------------------------------

        if grouping_columns:

            group_data = plot_data[
                [
                    f"_group_{column}"
                    for column in grouping_columns
                ]
            ]


            group_changed = (
                group_data
                .ne(group_data.shift())
                .any(axis=1)
            )


            boundaries = (
                group_changed
                .to_numpy()
                .nonzero()[0]
            )


            for boundary in boundaries[1:]:

                ax.axhline(
                    boundary,
                    color="white",
                    linewidth=10
                )


        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        plt.title(heatmap_title)

        plt.xlabel("Metric")

        plt.ylabel("Image ID")


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        plt.tight_layout()

        plt.savefig(
            heatmap_filename,
            dpi=300,
            bbox_inches="tight"
        )


        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        if show_plots:

            plt.show()


        plt.close()


        print(
            f"\nHeatmap saved to:\n{heatmap_filename}"
        )


    # ========================================================
    # 24. Generate image-quality heatmap
    # ========================================================

    save_heatmap(
        image_quality_norm,
        image_quality_metrics,
        image_quality_title,
        image_quality_heatmap,
        grouping_columns
    )


    # ========================================================
    # 25. Generate segmentation-quality heatmap
    # ========================================================

    save_heatmap(
        segmentation_norm,
        segmentation_heatmap_metrics,
        segmentation_title,
        segmentation_quality_heatmap,
        grouping_columns
    )


    # ========================================================
    # 26. Return results
    # ========================================================

    return (
        quality_and_segmentation,
        filtered_results,
        image_quality_norm,
        segmentation_norm
    )

In [ ]:
process_quality_results_filtered("Param_1")

In [ ]:
process_quality_results_filtered("Param_2", "current_nA")

In [ ]:
process_quality_results_filtered("Param_3")

In [ ]:
process_quality_results_filtered("Param_4", "magnification")

In [ ]:
process_quality_results_filtered("Param_5")

In [ ]:
process_quality_results_filtered("Param_6", "voltage_kV")

In [ ]:
process_quality_results_filtered("Param_7")

In [ ]:
process_quality_results_filtered("Param_8", filters={"voltage_kV": [5], "image_id": [66, 67, 68, 69]}, title="Parameter Set 8: ETD Image Quality Assessment Across Beam Currents",
filename="param_8_etd_image_quality_assessment_across_beam_currents")

In [ ]:
process_quality_results_filtered("Param_8", filters={"current_nA": [0.4], "image_id": [66, 80, 81, 85, 77]}, title="Parameter Set 8: ETD Image Quality Assessment Across Voltage at 0.4 nA",
filename="param_8_etd_image_quality_assessment_across_voltage_0.4nA")

In [ ]:
process_quality_results_filtered("Param_8", filters={"current_nA": [0.8], "image_id": [63, 70, 71, 72, 73]}, title="Parameter Set 8: ETD Image Quality Assessment Across Voltage at 0.8 nA",
filename="param_8_etd_image_quality_assessment_across_voltage_0.8nA")

In [ ]:
process_quality_results_filtered("Param_8", filters={"current_nA": [1.6], "image_id": [64, 78, 82, 83, 74]}, title="Parameter Set 8: ETD Image Quality Assessment Across Voltage at 1.6 nA",
filename="param_8_etd_image_quality_assessment_across_voltage_1.6nA")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

score_metrics = [
        "segmented_fraction",
        "mean_region_area",
        "median_region_area",
        "separability",
        "connected_components",
        "threshold_instability"
]
weights = {
    "segmented_fraction": 0.10,
    "mean_region_area": 0.15,
    "median_region_area": 0.15,
    "separability": 0.30,
    "connected_components": 0.15,
    "threshold_instability": 0.15
}


scaler = MinMaxScaler()


seg_results[
    score_metrics
] = scaler.fit_transform(
    seg_results[score_metrics]
)


seg_results["segmentation_score"] = (
    0.10 * seg_results["segmented_fraction"]
    + 0.15 * seg_results["mean_region_area"]
    + 0.15 * seg_results["median_region_area"]
    + 0.30 * seg_results["separability"]
    + 0.15 * (1 - seg_results["connected_components"])
    + 0.15 * (1 - seg_results["threshold_instability"])
)


seg_results.sort_values(
    "segmentation_score",
    ascending=False
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler


quality_metrics = [
    "Entropy",
    "Sharpness",
    "Gradient",
    "RMS_Contrast",
    "SNR"
]


# Make a copy to avoid modifying original data
quality_scored = quality_results.copy()


# Normalize metrics 0-1
scaler = MinMaxScaler()

quality_scored[
    quality_metrics
] = scaler.fit_transform(
    quality_scored[quality_metrics]
)


# Calculate image quality score
quality_scored["image_quality_score"] = (
    0.05 * quality_scored["Entropy"]
    +
    0.20 * quality_scored["Sharpness"]
    +
    0.25 * quality_scored["Gradient"]
    +
    0.20 * quality_scored["RMS_Contrast"]
    +
    0.30 * quality_scored["SNR"]
)


# Rank images
quality_scored = quality_scored.sort_values(
    "image_quality_score",
    ascending=False
)


quality_scored[
    [
        "image_id",
        "image_quality_score",
        "SNR",
        "Gradient",
        "Sharpness",
        "RMS_Contrast",
        "Entropy"
    ]
]

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


# -------------------------------
# Load results
# -------------------------------

quality_results = pd.read_csv(
    BASE_DIR / "FIB_SEM_optimization" / "results" / "image_quality_results.csv"
)
quality_results_normalised = pd.read_csv(
    BASE_DIR / "FIB_SEM_optimization" / "results" / "image_quality_results_normalised.csv"
)

seg_results = pd.read_csv(
    BASE_DIR / "FIB_SEM_optimization" / "results" / "Param_1" / "segmentation_quality_param_1.csv"
)


# -------------------------------
# Image Quality Score
# -------------------------------

quality_metrics = [
    "Entropy",
    "Sharpness",
    "Gradient",
    "RMS_Contrast",
    "SNR"
]


quality_scored_1 = quality_results.copy()

scaler = MinMaxScaler()

quality_scored_1[
    quality_metrics
] = scaler.fit_transform(
    quality_scored_1[quality_metrics]
)


quality_scored_1["image_quality_score"] = (
    0.05 * quality_scored_1["Entropy"]
    +
    0.20 * quality_scored_1["Sharpness"]
    +
    0.25 * quality_scored_1["Gradient"]
    +
    0.20 * quality_scored_1["RMS_Contrast"]
    +
    0.30 * quality_scored_1["SNR"]
)


# -------------------------------
# Segmentation Score
# -------------------------------

seg_metrics = [
    "separability",
    "connected_components",
    "mean_region_area"
]


seg_scored = seg_results.copy()


seg_scored[
    seg_metrics
] = scaler.fit_transform(
    seg_scored[seg_metrics]
)


seg_scored["segmentation_score"] = (
    0.40 * seg_scored["separability"]
    +
    0.20 * seg_scored["connected_components"]
    +
    0.20 * seg_scored["mean_region_area"]
    -
    0.20 * seg_scored["threshold_instability"]
)


# -------------------------------
# Combine Scores
# -------------------------------

combined_results = quality_scored.merge(
    seg_scored[
        [
            "detector",
            "sample_state",
            "material",
            "image_id",
            "segmentation_score"
        ]
    ],
    on=[
        "detector",
        "sample_state",
        "material",
        "image_id"
    ],
    how="inner"
)


# Final FIB-SEM optimisation score
combined_results["final_score"] = (
    0.4 * combined_results["image_quality_score"]
    +
    0.6 * combined_results["segmentation_score"]
)


# Rank best images
combined_results = combined_results.sort_values(
    "final_score",
    ascending=False
)


# Display ranking
combined_results[
    [
        "detector",
        "sample_state",
        "material",
        "image_id",
        "image_quality_score",
        "segmentation_score",
        "final_score"
    ]
]